<a href="https://colab.research.google.com/github/razanaaseeri-droid/Modern-Data-Engineering-for-AI-Systems/blob/main/Enterprise_Real_Time_Power_AI_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enterprise Real-Time Power AI Intelligence Platform

Pipeline stages (each prints output): Kafka ingest → schema validation + DLQ → quality gate → Bronze/Silver/Gold → embeddings + indexing → real retrieval + citations → Groq LLM analysis → evaluation → real-time dashboard.

Runtime: Runtime → Change runtime type → T4 GPU recommended.

## 1. Install dependencies

In [1]:
!pip install -q groq pandas numpy pydantic sentence-transformers kagglehub

## 2. Imports & logging

In [2]:
import os, uuid, time, logging, statistics
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pydantic import BaseModel, ValidationError
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("power_ai")
print("Logging initialized")

# ============================================================
# OUTPUT HELPER — prints a labeled summary + row count + sample

# ============================================================
def show_stage(title, data, sample_cols=None, n=3):
    """Print a boxed stage header, row count, and a small sample."""
    print("\n" + "="*68)
    print(f"  {title}")
    print("="*68)
    if isinstance(data, pd.DataFrame):
        print(f"  Rows: {len(data)}   Columns: {len(data.columns)}")
        cols = [c for c in (sample_cols or data.columns) if c in data.columns]
        if len(data):
            print("  Sample:")
            print(data[cols].head(n).to_string(index=False))
        else:
            print("  (empty)")
    elif isinstance(data, list):
        print(f"  Count: {len(data)}")
        for row in data[:n]:
            if isinstance(row, dict) and sample_cols:
                print("   -", {k: (str(row.get(k))[:40]) for k in sample_cols if k in row})
            else:
                print("   -", str(row)[:90])
    else:
        print("  ", data)
    print("-"*68)


Logging initialized


## 3. Groq API key (secure)


In [3]:
GROQ_API_KEY = None
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    pass
GROQ_API_KEY = GROQ_API_KEY or os.environ.get("GROQ_API_KEY")

GROQ_ENABLED = bool(GROQ_API_KEY)
print("Groq:", "ENABLED" if GROQ_ENABLED else "OFFLINE (no key) — extractive fallback will be used")

Groq: OFFLINE (no key) — extractive fallback will be used


## 4. Load & normalize the Kaggle dataset

In [4]:
import kagglehub, glob

DATASET_PATH = kagglehub.dataset_download("tobiasbueck/multilingual-customer-support-tickets")
csvs = sorted(glob.glob(f"{DATASET_PATH}/**/*.csv", recursive=True))
DATA_CSV = ([c for c in csvs if "20k" in c] or csvs)[0]
print("Using:", os.path.basename(DATA_CSV))

tickets_df = pd.read_csv(DATA_CSV)
print("Raw shape:", tickets_df.shape)
print("Columns:", list(tickets_df.columns))

# Normalize: rename to our internal names, fill missing
tickets_df = tickets_df.rename(columns={"body":"description","answer":"agent_response","queue":"section"})
tickets_df = tickets_df.fillna("unknown")

# Keep English + a workable sample so the demo runs fast and prints readable output
if "language" in tickets_df.columns:
    tickets_df = tickets_df[tickets_df["language"]=="en"]
tickets_df = tickets_df.head(50).reset_index(drop=True)
print("\nWorking sample:", tickets_df.shape)
display(tickets_df[["subject","description","agent_response","priority"]].head())

Using Colab cache for faster access to the 'multilingual-customer-support-tickets' dataset.
Using: dataset-tickets-multi-lang-4-20k.csv
Raw shape: (20000, 15)
Columns: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

Working sample: (50, 15)


,subject,description,agent_response,priority
0,Customer Support Inquiry,Seeking information on digital strategies that...,We offer a variety of digital strategies and s...,medium
1,Data Analytics for Investment,I am contacting you to request information on ...,I am here to assist you with data analytics to...,medium
2,Security,"Dear Customer Support, I am reaching out to in...","Dear [name], we take the security of medical d...",medium
3,Concerns About Securing Medical Data on 2-in-1...,Inquiring about best practices for securing me...,Thank you for your concern regarding securing ...,medium
4,Problem with Integration,"The integration stopped working unexpectedly, ...",I will look into the problem and call you at <...,high


## 5. Deliverable 1 — Ingestion: Mock Kafka
Here it's a proper class with working `produce`/`consume`,
and we verify it exists.

In [5]:
class MockKafka:
    """Simulates a Kafka topic/partition/offset model in memory."""
    def __init__(self):
        self.topics = {}

    def create_topic(self, name):
        self.topics.setdefault(name, [])

    def produce(self, topic, message):
        self.create_topic(topic)
        offset = len(self.topics[topic])
        self.topics[topic].append({
            "offset": offset,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "message": message,
        })
        return offset

    def consume(self, topic, last_offset=0):
        return self.topics.get(topic, [])[last_offset:]

mock_kafka = MockKafka()
print("MockKafka initialized. mock_kafka in globals():", "mock_kafka" in globals())

MockKafka initialized. mock_kafka in globals(): True


## 6. JSON schema validation + Dead-Letter Queue + retry

In [6]:
class TicketSchema(BaseModel):
    subject: str
    priority: str
    language: str
    description: str

DLQ_TOPIC = "dead_letter_queue"
VALID_TOPIC = "tickets.valid"

def safe_produce(message, max_retries=2):
    """Validate against schema; route valid -> VALID_TOPIC, invalid -> DLQ. Retries transient errors."""
    for attempt in range(max_retries + 1):
        try:
            TicketSchema(
                subject=str(message.get("subject","unknown")),
                priority=str(message.get("priority","low")),
                language=str(message.get("language","unknown")),
                description=str(message.get("description","unknown")),
            )
            mock_kafka.produce(VALID_TOPIC, message)
            return True
        except ValidationError as e:
            mock_kafka.produce(DLQ_TOPIC, {"error": str(e), "payload": message})
            return False
        except Exception as e:
            if attempt < max_retries:
                time.sleep(0.05); continue
            mock_kafka.produce(DLQ_TOPIC, {"error": f"retry_exhausted: {e}", "payload": message})
            return False

print("Schema + DLQ + retry ready.")

Schema + DLQ + retry ready.


## 7. Deliverable 5 — Quality Gate (with real scoring)

In [7]:
quarantine_records = []

def quality_score(ticket):
    """0-100 score from concrete rules (not a placeholder)."""
    score = 100
    desc = str(ticket.get("description",""))
    subj = str(ticket.get("subject",""))
    if len(desc) < 10:               score -= 25   # too-short body
    if subj.strip() in ("","unknown"): score -= 40 # missing subject
    if str(ticket.get("priority","")).lower() not in ("low","medium","high"): score -= 15
    return max(score, 0)

def quality_gate(ticket):
    s = quality_score(ticket)
    if s < 70:
        quarantine_records.append({"ticket": ticket, "score": s})
        return False, s
    return True, s

print("Quality gate ready.")

Quality gate ready.


## 8. Deliverable 2 — Delta Lakehouse (Bronze / Silver / Gold)

In [8]:
# Bronze = raw immutable | Silver = cleaned/deduped | Gold = analytics aggregate
bronze_table, silver_table = [], []
gold_table = pd.DataFrame()

def write_bronze(ticket):
    bronze_table.append(dict(ticket))                       # raw, immutable copy

def merge_silver(ticket):
    # cleaned + enriched record; dedupe happens after the loop
    silver_table.append({
        **ticket,
        "is_resolved": str(ticket.get("agent_response","unknown")) != "unknown",
        "desc_len": len(str(ticket.get("description",""))),
    })

def build_gold():
    global gold_table
    if not silver_table:
        return
    df = pd.DataFrame(silver_table).drop_duplicates(subset=["subject","description"])
    gold_table = (df.groupby("priority").size()
                    .reset_index(name="ticket_count")
                    .sort_values("ticket_count", ascending=False))

print("Lakehouse ready (Bronze/Silver/Gold).")

Lakehouse ready (Bronze/Silver/Gold).


## 9. Deliverable 3 — Embeddings, indexing & REAL retrieval

The original `retrieve()` ignored the query. This version encodes the query and ranks the
vector store by **cosine similarity**, so results actually depend on what's asked.

In [9]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
vector_store = []
print("Embedding model loaded.")

def index_ticket(ticket, tid):
    """Index the agent_response (the knowledge) with metadata for citation."""
    text = f"{ticket.get('subject','')}. {ticket.get('agent_response','')}".strip()
    if len(text) < 15:
        return
    emb = embedding_model.encode(text, normalize_embeddings=True)
    vector_store.append({
        "id": f"KB-{tid}", "text": text,
        "subject": ticket.get("subject",""), "priority": ticket.get("priority",""),
        "embedding": emb,
    })

def retrieve(query, k=3):
    """REAL cosine-similarity search (was: return vector_store[:3])."""
    if not vector_store: return []
    qv = embedding_model.encode(query, normalize_embeddings=True)
    scored = sorted(vector_store, key=lambda v: float(np.dot(qv, v["embedding"])), reverse=True)
    return [{"id":v["id"], "text":v["text"], "subject":v["subject"],
             "score":round(float(np.dot(qv, v["embedding"])),3)} for v in scored[:k]]

def cite_sources(results):
    return [{"marker":i+1, "id":r["id"], "subject":r["subject"], "score":r["score"]}
            for i,r in enumerate(results)]

print("Real retrieval + citation ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Real retrieval + citation ready.


## 10. Governance — OpenLineage + audit (FIXED datetime import)

In [10]:
lineage_events = []
audit_events = []

def emit_lineage(stage, inputs=None, outputs=None):
    lineage_events.append({"stage":stage, "inputs":inputs or [], "outputs":outputs or [],
                           "timestamp": datetime.now(timezone.utc).isoformat()})

def audit(stage, action, status="SUCCESS", **details):
    audit_events.append({"stage":stage, "action":action, "status":status,
                         "timestamp": datetime.now(timezone.utc).isoformat(), **details})

print("Lineage + audit ready.")

Lineage + audit ready.


## 11. Deliverable — Groq LLM analysis (with offline fallback)

In [11]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_ENABLED else None
GROQ_MODEL    = "llama-3.1-8b-instant"
GROQ_FALLBACK = "openai/gpt-oss-20b"

def groq_analyze(query, retrieved):
    """Generate a cited answer from retrieved context. Falls back gracefully."""
    context = "\n\n".join(f"[{i+1}] {r['text']}" for i,r in enumerate(retrieved))
    if not GROQ_ENABLED:
        top = retrieved[0]["text"] if retrieved else "no context"
        return f"[offline] Based on the top source: {top[:160]}... [1]"
    system = ("You are an enterprise support-assist AI. Answer ONLY from the numbered excerpts. "
              "Cite claims like [1], [2]. If they don't answer it, say so.")
    user = f"Customer question:\n{query}\n\nExcerpts:\n{context}\n\nWrite a concise cited answer."
    for model in (GROQ_MODEL, GROQ_FALLBACK):
        try:
            resp = groq_client.chat.completions.create(
                model=model, temperature=0, max_tokens=150,
                messages=[{"role":"system","content":system},{"role":"user","content":user}])
            return resp.choices[0].message.content
        except Exception as e:
            if model == GROQ_FALLBACK:
                return f"[generation error: {e}]"
    return "[unreachable]"

print("Groq module ready.")

Groq module ready.


## 12. Evaluation — retrieval quality & groundedness

In [12]:
def precision_at_k(retrieved, relevant_ids, k=3):
    top = [r["id"] for r in retrieved[:k]]
    return round(len(set(top)&set(relevant_ids))/max(len(top),1), 3)

def groundedness(answer, retrieved):
    """Does the answer reference the retrieved sources? Checks citation markers + overlap."""
    ans = str(answer).lower()
    has_citation = any(f"[{i+1}]" in answer for i in range(len(retrieved)))
    overlap = any(any(w in ans for w in r["subject"].lower().split()[:3]) for r in retrieved if r["subject"])
    return has_citation or overlap

print("Evaluation ready.")

Evaluation ready.


## 13. MAIN EXECUTION

This loop drives every stage on
real tickets and prints results. We inject 2 deliberately-bad tickets to prove the DLQ +
quarantine work.

In [13]:
records = tickets_df.to_dict(orient="records")

# Inject 2 deliberately-bad tickets to prove DLQ + quality quarantine work
records += [
    {"subject":"", "description":"x", "priority":"weird", "language":"en", "agent_response":"unknown"},      # schema/quality fail
    {"subject":"Login issue", "description":"short", "priority":"high", "language":"en", "agent_response":"unknown"},  # low quality
]

emit_lineage("ingest", inputs=["kaggle"], outputs=["tickets.valid","dead_letter_queue"])

# ---- counters so we can report per stage ----
n_produced = n_valid = n_dlq = n_quarantined = n_bronze = n_indexed = 0

print("\n" + "#"*68)
print("#  RUNNING ENTERPRISE PIPELINE — stage-by-stage output")
print("#"*68)

for i, ticket in enumerate(records):
    n_produced += 1
    # STAGE 1: schema validation -> valid topic or DLQ
    if not safe_produce(ticket):
        n_dlq += 1
        audit("ingestion","QUARANTINED_DLQ",status="WARNING")
        continue
    n_valid += 1
    # STAGE 2: quality gate
    passed, score = quality_gate(ticket)
    if not passed:
        n_quarantined += 1
        audit("quality","QUARANTINED",status="WARNING",score=score)
        continue
    # STAGE 3: lakehouse bronze + silver + index
    write_bronze(ticket); n_bronze += 1
    merge_silver(ticket)
    index_ticket(ticket, i); n_indexed += 1

build_gold()
emit_lineage("lakehouse", inputs=["tickets.valid"], outputs=["bronze","silver","gold"])
emit_lineage("index", inputs=["silver"], outputs=["vector_store"])
audit("pipeline","RUN_COMPLETE", processed=n_valid, indexed=n_indexed,
      dlq=n_dlq, quarantined=n_quarantined)

# ============================================================
# PER-STAGE OUTPUT — see the data + counts after each step
# ============================================================

# --- INGESTION ---
show_stage("STAGE 1 — INGESTION (Kafka)",
           mock_kafka.consume("tickets.valid"),
           sample_cols=["offset","timestamp"])
print(f"  Produced total : {n_produced}")
print(f"  Valid (schema) : {n_valid}")
print(f"  DLQ (rejected) : {n_dlq}")
if n_dlq:
    print("  DLQ sample:")
    for d in mock_kafka.consume(DLQ_TOPIC)[:2]:
        print("     -", str(d["message"].get("error",""))[:70])

# --- QUALITY GATE ---
show_stage("STAGE 2 — QUALITY GATE", quarantine_records, sample_cols=["score"])
print(f"  Passed quality : {n_valid - n_quarantined}")
print(f"  Quarantined    : {n_quarantined}")
if quarantine_records:
    print("  Quarantined sample (subject | score):")
    for q in quarantine_records[:3]:
        print(f"     - '{q['ticket'].get('subject','')[:30]}' | score={q['score']}")

# --- BRONZE ---
bronze_df = pd.DataFrame(bronze_table)
show_stage("STAGE 3 — BRONZE (raw immutable)", bronze_df,
           sample_cols=["subject","priority","language"])

# --- SILVER ---
silver_df = pd.DataFrame(silver_table)
show_stage("STAGE 4 — SILVER (cleaned + enriched)", silver_df,
           sample_cols=["subject","priority","is_resolved","desc_len"])
print(f"  Bronze rows : {len(bronze_df)}")
print(f"  Silver rows : {len(silver_df)}  (after enrichment)")

# --- GOLD ---
show_stage("STAGE 5 — GOLD (analytics aggregate: tickets by priority)", gold_table)
print(f"  Total tickets across priorities : {int(gold_table['ticket_count'].sum()) if len(gold_table) else 0}")

# --- VECTOR INDEX ---
show_stage("STAGE 6 — VECTOR INDEX (Qdrant-style store)", vector_store,
           sample_cols=["id","subject","priority"])
print(f"  Vectors indexed : {n_indexed}")
print(f"  Embedding dim   : {len(vector_store[0]['embedding']) if vector_store else 0}")

# --- ROW-COUNT SUMMARY TABLE ---
print("\n" + "="*68)
print("  PIPELINE ROW-COUNT SUMMARY")
print("="*68)
summary = pd.DataFrame([
    {"stage":"1. Produced (raw)",      "rows": n_produced},
    {"stage":"2. Valid (schema pass)", "rows": n_valid},
    {"stage":"   -> DLQ (schema fail)","rows": n_dlq},
    {"stage":"3. Quality passed",      "rows": n_valid - n_quarantined},
    {"stage":"   -> Quarantined",      "rows": n_quarantined},
    {"stage":"4. Bronze",              "rows": len(bronze_df)},
    {"stage":"5. Silver",              "rows": len(silver_df)},
    {"stage":"6. Gold (priority grps)","rows": len(gold_table)},
    {"stage":"7. Vectors indexed",     "rows": n_indexed},
])
print(summary.to_string(index=False))


####################################################################
#  RUNNING ENTERPRISE PIPELINE — stage-by-stage output
####################################################################

  STAGE 1 — INGESTION (Kafka)
  Count: 52
   - {'offset': '0', 'timestamp': '2026-07-03T23:17:22.852509+00:00'}
   - {'offset': '1', 'timestamp': '2026-07-03T23:17:23.121145+00:00'}
   - {'offset': '2', 'timestamp': '2026-07-03T23:17:23.132115+00:00'}
--------------------------------------------------------------------
  Produced total : 52
  Valid (schema) : 52
  DLQ (rejected) : 0

  STAGE 2 — QUALITY GATE
  Count: 4
   - {'score': '60'}
   - {'score': '60'}
   - {'score': '60'}
--------------------------------------------------------------------
  Passed quality : 48
  Quarantined    : 4
  Quarantined sample (subject | score):
     - 'unknown' | score=60
     - 'unknown' | score=60
     - 'unknown' | score=60

  STAGE 3 — BRONZE (raw immutable)
  Rows: 48   Columns: 15
  Sample:
            

### 13.1 Retrieval + Groq answer on live queries

In [14]:
demo_queries = [
    "How do I reset my password?",
    "Where is my refund and how long does it take?",
    "My payment keeps getting declined",
]

for q in demo_queries:
    print("="*66); print("QUERY:", q)
    results = retrieve(q, k=3)
    cites = cite_sources(results)
    print("\nTop retrieved (real cosine scores):")
    for c in cites:
        print(f"   [{c['marker']}] {c['id']} — {c['subject'][:45]}  (score {c['score']})")
    answer = groq_analyze(q, results)
    print("\nGROQ ANSWER:\n ", answer)
    # eval on the fly
    p = precision_at_k(results, [results[0]["id"]] if results else [], k=3)
    g = groundedness(answer, results)
    print(f"\n  precision@3={p} | grounded={g}")
    audit("rag","ANSWER", status="SUCCESS", grounded=g)

QUERY: How do I reset my password?

Top retrieved (real cosine scores):
   [1] KB-51 — Login issue  (score 0.456)
   [2] KB-33 — Support Request for Data Breach Issue  (score 0.274)
   [3] KB-47 — Support Request: Medical Data Encryption Issu  (score 0.223)

GROQ ANSWER:
  [offline] Based on the top source: Login issue. unknown... [1]

  precision@3=0.333 | grounded=True
QUERY: Where is my refund and how long does it take?

Top retrieved (real cosine scores):
   [1] KB-22 — Problem with Invoice  (score 0.228)
   [2] KB-33 — Support Request for Data Breach Issue  (score 0.199)
   [3] KB-41 — Concerns with Data Analytics  (score 0.142)

GROQ ANSWER:
  [offline] Based on the top source: Problem with Invoice. In response to your email about the incorrect total on your invoice, I want to acknowledge the inconvenience this has caused you. To bette... [1]

  precision@3=0.333 | grounded=True
QUERY: My payment keeps getting declined

Top retrieved (real cosine scores):
   [1] KB-22 — Problem w

## 14. Deliver real-time intelligence — live dashboard

Wraps query serving with latency measurement and a rolling dashboard (avg/p95 latency,
grounded-rate, throughput) plus SLA alerts — the observability layer that makes this a
real-time service, matching the Groq-upgrade notebook.

In [15]:
class RealTimeIntelligence:
    def __init__(self, p95_sla_ms=3000, grounded_floor=0.6):
        self.records=[]; self.p95_sla=p95_sla_ms; self.grounded_floor=grounded_floor
    def ask(self, query):
        t0=time.perf_counter()
        results=retrieve(query,k=3)
        answer=groq_analyze(query,results)
        latency=(time.perf_counter()-t0)*1000
        g=groundedness(answer,results)
        self.records.append({"latency_ms":round(latency,1),"grounded":g,"n_cites":len(results)})
        audit("realtime","QUERY_SERVED",status="SUCCESS" if g else "WARNING",latency_ms=round(latency,1))
        return {"query":query,"answer":answer,"latency_ms":round(latency,1),"grounded":g}
    def dashboard(self):
        if not self.records: return {}
        lats=sorted(r["latency_ms"] for r in self.records)
        p95=lats[max(0,int(len(lats)*0.95)-1)]
        grounded_rate=sum(r["grounded"] for r in self.records)/len(self.records)
        alerts=[]
        if p95>self.p95_sla: alerts.append(f"p95 latency {p95:.0f}ms > SLA {self.p95_sla}ms")
        if grounded_rate<self.grounded_floor: alerts.append(f"grounded-rate {grounded_rate:.0%} < floor {self.grounded_floor:.0%}")
        return {"queries":len(self.records),"avg_latency_ms":round(statistics.mean(lats),1),
                "p95_latency_ms":round(p95,1),"grounded_rate":round(grounded_rate,3),
                "alerts":alerts or ["none"]}

rt = RealTimeIntelligence()
print("Serving live query stream...\n")
for q in demo_queries + ["how do I enable two factor authentication","cancel my subscription"]:
    r=rt.ask(q)
    tag="" if r["grounded"] else "  [NOT GROUNDED]"
    print(f"  {r['latency_ms']:7.1f} ms | grounded={r['grounded']}{tag} | {q[:45]}")

import json
print("\n"+"="*50); print("REAL-TIME DASHBOARD"); print("="*50)
print(json.dumps(rt.dashboard(), indent=2))

Serving live query stream...

     10.5 ms | grounded=True | How do I reset my password?
      7.0 ms | grounded=True | Where is my refund and how long does it take?
      6.9 ms | grounded=True | My payment keeps getting declined
      6.9 ms | grounded=True | how do I enable two factor authentication
      6.9 ms | grounded=True | cancel my subscription

REAL-TIME DASHBOARD
{
  "queries": 5,
  "avg_latency_ms": 7.6,
  "p95_latency_ms": 7.0,
  "grounded_rate": 1.0,
  "alerts": [
    "none"
  ]
}


### 14.1 Governance trail

In [16]:
print("LINEAGE EVENTS:")
for e in lineage_events:
    print(f"  {e['stage']:10} {e['inputs']} -> {e['outputs']}")

print(f"\nAUDIT EVENTS: {len(audit_events)} total")
adf = pd.DataFrame(audit_events)
print(adf["stage"].value_counts().to_string())

LINEAGE EVENTS:
  ingest     ['kaggle'] -> ['tickets.valid', 'dead_letter_queue']
  lakehouse  ['tickets.valid'] -> ['bronze', 'silver', 'gold']
  index      ['silver'] -> ['vector_store']

AUDIT EVENTS: 13 total
stage
realtime    5
quality     4
rag         3
pipeline    1
